In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8cd319b4f2187b6b29bb69603a96460fc325a975"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8cd319b4f2187b6b29bb69603a96460fc325a975/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Custom Layers and Functions

The library ships over a hundred layers, yet every one of them started life as
code somebody wrote because the layer they needed did not exist. Sooner or
later you will be in the same position: a new normalization, an unusual
residual block, an operation with no gradient. This section shows what to do.
A custom layer is a subclass of the module class from
that section with a forward method, and if you
register its state properly you inherit everything a built-in layer gets:
parameter tracking, serialization, and device movement, with no extra code. We
build up from a stateless five-liner to RMSNorm, a normalization used by many
current language models, then to layers with precomputed non-trainable state,
and finally to the case where the forward computation alone is not enough
because the gradient itself must be redefined.

In [1]:
import jax
from jax import numpy as jnp
from flax import nnx
from flax import serialization
from d2l import jax as d2l

## Layers without Parameters

The smallest custom layer has no state at all. `CenteredLayer` subtracts the
mean from its input. To build it, we inherit from the base module class and
implement `__call__`. This layer owns no variables and needs no constructor.

In [2]:
class CenteredLayer(nnx.Module):
    def __call__(self, X):
        return X - X.mean()

Feeding data through confirms that it does what it says.

NNX modules are ordinary Python objects: calling the object invokes
`__call__` directly. Since `CenteredLayer` owns no variables, construction
needs neither an input shape nor an RNG stream.

In [3]:
layer = CenteredLayer()
layer(jnp.array([1.0, 2, 3, 4, 5]))

Array([-2., -1.,  0.,  1.,  2.], dtype=float32)

Nothing distinguishes this class from a built-in layer. We can place it inside
a `Sequential`, and the container neither knows nor cares that one of its
children is user code. The output mean should be zero; because we are adding
up floating-point numbers, we may see a very small nonzero value instead,
which is roundoff, not a bug.

NNX creates the `Linear` parameters in its constructor, so we specify both
feature dimensions there. `CenteredLayer` contributes no parameters.

In [4]:
net = nnx.Sequential(nnx.Linear(8, 128, rngs=nnx.Rngs(0)), CenteredLayer())
Y = net(jax.random.uniform(d2l.get_key(), (4, 8)))
Y.mean()

Array(-2.3283064e-09, dtype=float32)

## Layers with Parameters: RMSNorm

A layer with something to learn must create its own parameters, and
`nnx.Param` is what registers them in the object graph
(that section). We could show the mechanics by re-implementing
`nnx.Linear`, but that teaches nothing the built-in does not already do. Instead
we implement *RMSNorm* [@Zhang.Sennrich.2019], a normalization used by
many current language models, in the same handful of lines.

Layer normalization standardizes each input vector: subtract the mean, divide
by the standard deviation, then apply a learned scale and shift. Zhang and
Sennrich observed that the re-centering contributes little and dropped it,
along with the shift. What remains is: divide by the root mean square,
multiply by a learned gain $\mathbf{g}$:

$$
\textrm{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\sqrt{\frac{1}{d} \sum_{i=1}^{d} x_i^2 + \epsilon}} \odot \mathbf{g},
$$

where $d$ is the width of $\mathbf{x}$ and $\epsilon$ guards against division
by zero. One parameter vector, one reduction, no shift. Why dropping the mean
costs so little in practice is a question for the normalization discussion in
later chapters; here it is our stateful worked example.

In [5]:
class RMSNorm(nnx.Module):
    def __init__(self, d, eps=1e-6):
        self.gain = nnx.Param(jnp.ones(d))
        self.eps = eps

    def __call__(self, X):
        rms = jnp.sqrt((X ** 2).mean(-1, keepdims=True) + self.eps)
        return self.gain * X / rms

Let $m_2$ denote an input row's mean square. With gain one, the output mean
square is $m_2/(m_2 + \epsilon)$. It is therefore close to one when
$m_2 \gg \epsilon$, as the badly scaled inputs below verify.

In [6]:
norm = RMSNorm(8)
X = 100 * jax.random.normal(d2l.get_key(), (4, 8))
(norm(X) ** 2).mean(-1)

Array([1.       , 0.9999998, 1.0000001, 1.0000001], dtype=float32)

### The Composability Guarantee

The reason to write RMSNorm as a module, rather than as a function with a
gain tensor floating around beside it, is what registration buys. A correctly
written custom layer is indistinguishable from a built-in one along four
axes: its parameters are tracked, it composes inside containers, its state
serializes, and it moves across devices. We check each once.

First, assigning `nnx.Param` registered the gain in the NNX object graph.
`nnx.state(norm, nnx.Param)` selects it, and an `nnx.Optimizer` configured
with `wrt=nnx.Param` will update it.

In [7]:
nnx.state(norm, nnx.Param)

State({
  'gain': Param( # 8 (32 B)
    value=Array([1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)
  )
})

Second, the layer drops into a `Sequential` next to built-ins.

In [8]:
net = nnx.Sequential(nnx.Linear(20, 8, rngs=nnx.Rngs(0)), RMSNorm(8),
                     nnx.Linear(8, 2, rngs=nnx.Rngs(1)))
Y = net(jax.random.normal(d2l.get_key(), (4, 20)))
Y.shape

(4, 2)

Third, its state serializes with everything else. `flax.serialization` turns
the pure dictionary view of NNX state into bytes and back; restoring those
values into the state of a fresh model makes the two models agree exactly
(saving to disk works the same way; see that section).

In [9]:
graph, state = nnx.split(net)
raw = serialization.to_bytes(nnx.to_pure_dict(state))
X = jax.random.normal(d2l.get_key(), (4, 20))
clone = nnx.Sequential(nnx.Linear(20, 8, rngs=nnx.Rngs(2)), RMSNorm(8),
                       nnx.Linear(8, 2, rngs=nnx.Rngs(3)))
clone_graph, clone_state = nnx.split(clone)
restored = serialization.from_bytes(nnx.to_pure_dict(clone_state), raw)
nnx.replace_by_pure_dict(clone_state, restored)
clone = nnx.merge(clone_graph, clone_state)
bool(jnp.array_equal(net(X), clone(X)))

True

Fourth, it moves. NNX state is a pytree of `Variable` objects whose array
values can be placed on a chosen device and written back with `nnx.update`.
On a machine with a GPU (JAX's default device there) the cell below reports a
GPU device twice; on a CPU-only machine it reports `cpu` and runs just the
same.

In [10]:
device = jax.devices()[0]
nnx.update(net, jax.device_put(nnx.state(net), device))
net.layers[1].gain.device, net(X).device

(CudaDevice(id=0), CudaDevice(id=0))

None of this took any code beyond the class definition. The guarantee comes
from the base class: subclass it, register state through the proper channels,
and containers, optimizers, checkpoints, and devices all treat your layer as
native.

### Checking against the Built-in
<!-- d2l:prose id=custom-layers-md-18 fw=pytorch, jax, tensorflow -->

RMSNorm proved useful enough that the library now ships its own
implementation. That gives us a referee. We copy a nontrivial gain into both
implementations, so that agreement cannot be an accident of default values,
and compare outputs.

In [11]:
ours = RMSNorm(8)
native = nnx.RMSNorm(8, epsilon=1e-6, rngs=nnx.Rngs(0))
gain = jnp.linspace(0.5, 1.5, 8)
X = jax.random.normal(d2l.get_key(), (16, 8))
ours.gain[...] = gain
native.scale[...] = gain
bool(jnp.allclose(ours(X), native(X)))

True

## Precomputed State: Buffers

Some layer state is neither a parameter nor a passing activation: a causal
mask, a fixed positional table, running statistics. Such values must
persist, serialize, and move to the device together with the model — yet no
optimizer may ever touch them. Frameworks give this third kind of state its
own channel, and custom layers are where you create it.

that section introduced state that persists and travels with
the model but receives no gradient. NNX distinguishes kinds of state through
`Variable` subclasses. `nnx.Param` marks trainable state; a custom subclass
can mark a buffer that an optimizer must skip. A common example is the causal
mask that keeps attention scores from looking at future positions. The mask
is fixed, so a parameter is wrong. Storing it in `Buffer` keeps it out of the
gradient computation while it still serializes and moves with the model.
BatchNorm uses the same idea for its running statistics through
`nnx.BatchStat`.

One caveat before the code: the compact mask below accepts square
self-attention score matrices. Cached decoding, where query and key lengths
differ, needs an offset mask rather than this $T \times T$ slice.

In [12]:
class Buffer(nnx.Variable):
    pass

class CausalMask(nnx.Module):
    def __init__(self, max_len):
        self.mask = Buffer(jnp.triu(
            jnp.ones((max_len, max_len), dtype=bool), 1))

    def __call__(self, scores):
        T = scores.shape[-1]
        return jnp.where(self.mask[:T, :T], -jnp.inf, scores)

The layer masks the strict upper triangle, has no `nnx.Param` variables for
the optimizer, and still carries the mask as `Buffer` state.

In [13]:
mask = CausalMask(max_len=8)
print(mask(jnp.zeros((3, 3))))
print(nnx.state(mask, nnx.Param), nnx.state(mask, Buffer))

[[  0. -inf -inf]
 [  0.   0. -inf]
 [  0.   0.   0.]]
State({}) State({
  'mask': Buffer( # 64 (64 B)
    value=Array([[False,  True,  True,  True,  True,  True,  True,  True],
           [False, False,  True,  True,  True,  True,  True,  True],
           [False, False, False,  True,  True,  True,  True,  True],
           [False, False, False, False,  True,  True,  True,  True],
           [False, False, False, False, False,  True,  True,  True],
           [False, False, False, False, False, False,  True,  True],
           [False, False, False, False, False, False, False,  True],
           [False, False, False, False, False, False, False, False]],      dtype=bool)
  )
})


## Custom Gradients

So far we customized the forward computation and let automatic
differentiation derive the backward pass. That derivation is literal: it
differentiates exactly the operations you ran. Occasionally literalness is
the problem. Quantization rounds values to a grid, and rounding is flat
almost everywhere, so its true derivative is zero. Any parameter sitting
behind a rounding operation stops learning:

In [14]:
w = jnp.array([0.9, 1.4, 2.6])
jax.grad(lambda w: jnp.round(w).sum())(w)

Array([0., 0., 0.], dtype=float32)

Zero gradient, exactly as calculus demands, and useless for training. The
*straight-through estimator* [@Bengio.Leonard.Courville.2013] resolves
this with a controlled lie: keep the rounding in the forward pass, but
pretend in the backward pass that it was the identity.

No automatic system can derive a lie for us, so we override the chain rule
with `jax.custom_vjp`, attaching a hand-written backward rule (a
vector-Jacobian product) to an ordinary function, the split
the figure draws explicitly.

![The straight-through estimator, forward versus backward. Forward keeps the true staircase round(x), close to but not the same as the identity it approximates; backward substitutes a constant surrogate gradient of 1, the identity's own derivative, for the true gradient, which is zero almost everywhere and would stop all learning.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-ste.svg)

In [15]:
@jax.custom_vjp
def round_ste(X):
    return jnp.round(X)

def round_ste_fwd(X):
    return jnp.round(X), None  # (output, residuals); we need no residuals

def round_ste_bwd(residuals, grad_output):
    # Pretend forward was the identity: pass the gradient straight through
    return (grad_output,)

round_ste.defvjp(round_ste_fwd, round_ste_bwd)

Two rules govern the definition. First, the forward rule must return a
*pair*: the output plus whatever residuals the backward rule will need. Ours
needs nothing and returns `None`; returning the output alone is the most
common first-time bug, and JAX reports it as a puzzling structure mismatch
rather than pointing at the missing residuals. Second, the backward rule
receives those residuals and the loss gradient with respect to the output,
and must return a tuple holding one gradient per argument of the original
function, hence `(grad_output,)` rather than `grad_output`. An argument that
is not differentiable data, say a configuration flag, must be declared
through `nondiff_argnums` so it is routed past the gradient machinery.

A small example verifies that the gradient now flows. We multiply the rounded
values by known weights, so we know what gradient to expect at `w`.

In [16]:
jax.grad(lambda w: (round_ste(w) * jnp.array([1.0, 2.0, 3.0])).sum())(w)

Array([1., 2., 3.], dtype=float32)

The downstream gradient arrives at `w` untouched, as if the rounding were the
identity, while the forward pass still computed with rounded values. Some
estimators instead zero the gradient where a saved input had saturated, while
ordinary gradient clipping bounds a large upstream gradient independently of
the quantizer. These are different approximations. Neither is a universal
default; the surrogate backward rule must match the training objective it is
meant to approximate.

JAX offers a shortcut for this particular lie. `jax.lax.stop_gradient` is an
identity whose gradient is zero, so `X + stop_gradient(round(X) - X)`
computes `round(X)` in the forward pass while the backward pass sees only the
leading `X`. Three lines, no `custom_vjp`; the general mechanism earns its
keep when the surrogate is not the identity, an input-dependent saturation
rule for instance. The two definitions agree in value and in gradient:

In [17]:
def round_ste2(X):
    return X + jax.lax.stop_gradient(jnp.round(X) - X)

def loss(f, w):
    return (f(w) * jnp.array([1.0, 2.0, 3.0])).sum()

(bool(jnp.allclose(round_ste(w), round_ste2(w))),
 bool(jnp.allclose(jax.grad(loss, 1)(round_ste, w),
                   jax.grad(loss, 1)(round_ste2, w))))

(True, True)

A scope note to close. A custom gradient redefines the gradient of a
computation assembled from existing tensor operations; it does not create new
ones. Writing new *kernels*, code that runs on the accelerator itself, is a
separate craft that we do not cover in this book; that section
discusses how to get performance out of the operations you already have.

## Summary

A custom layer is a module subclass: `__call__` defines the computation,
`nnx.Param` registers learnable state, and another `nnx.Variable` subclass
registers persistent state that the optimizer does not touch. Registration
is what buys composability; a correctly written layer
gets parameter tracking, container compatibility, serialization, and device
movement for free, as we verified on RMSNorm axis by axis. When the chain
rule itself must be overridden, as in the straight-through estimator,
`jax.custom_vjp` attaches a forward and a backward rule to a function, and
`jax.lax.stop_gradient` covers the identity-surrogate case in a single
expression. Build custom implementations to understand them; prefer the
native ones in production.

## Exercises

1. Add an optional learned bias to `RMSNorm` (a shift applied after the
   scale, restoring part of what LayerNorm had). Verify that the parameter
   state of a model containing it grows by the expected entry, and observe
   what `flax.serialization.from_bytes` does when it restores bytes saved
   without the bias into the new state structure.
1. Implement `Dropout` from scratch as a custom layer that zeroes each entry
   with probability $p$ and rescales the survivors during training, but is
   the identity during evaluation. Store `self.deterministic = False`, give
   the module an `nnx.Rngs` stream, and implement
   `set_view(self, *, deterministic)` to update the flag. Verify that
   `nnx.view(model, deterministic=True)` reaches a dropout layer nested in an
   `nnx.Sequential`.
1. Implement a clamp with a custom gradient: a `custom_vjp` function whose
   forward is `jnp.clip(X, lo, hi)` and whose backward passes the gradient
   only where the input lay strictly inside the clamp range; the forward rule
   must save `X` as a residual. Compare its gradients with those of the
   native `jnp.clip` for inputs inside, outside, and exactly on the boundary.
   Which convention does the native operation use at the boundary?
1. Write a layer that returns the leading half of the Fourier coefficients of
   its input. It has no parameters, so nothing registers. What do you still
   gain by wrapping the computation in a module instead of calling the
   transform inline wherever you need it?